In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        
        # 1. 첫 번째 합성곱 층 (Convolution Layer)
        
        # 입력 채널: 1 (흑백 이미지), 출력 채널: 6, 커널 크기: 3x3, 패딩: 1 (28x28 크기 유지)
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=3, stride=1, padding=1)
        
        # 2. 첫 번째 풀링 층 (Pooling Layer: 28x28 -> 14x14)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # 3. 두 번째 합성곱 층 (Convolution Layer)
        # 입력 채널: 6, 출력 채널: 16, 커널 크기: 3x3, 패딩: 1 (14x14 크기 유지)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=3, stride=1, padding=1)
        
        # 4. 두 번째 풀링 층 (Pooling Layer: 14x14 -> 7x7)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # 5. 완전 연결 층 (Fully Connected / Linear Layers)
        # 최종 특성 맵 크기: 16채널 * 7 * 7 = 784
        self.fc1 = nn.Linear(16 * 7 * 7, 120)
        self.fc2 = nn.Linear(120, 10)  # 최종 출력 클래스 개수: 10개 (0~9 숫자 분류)

    def forward(self, x):
        # [특성 추출 부 - Feature Extraction]
        # 입력 이미지 -> Conv1 -> ReLU -> MaxPool1 (28x28 -> 14x14)
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool1(x)
        
        # Conv2 -> ReLU -> MaxPool2 (14x14 -> 7x7)
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool2(x)
        
        # 선형 결합(Fully Connected)을 위해 차원 펼치기 (Flatten)
        x = x.view(-1, 16 * 7 * 7)
        
        # [분류 부 - Classification]
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        
        # Softmax 활성화 함수 적용 (확률 값 출력)
        output = F.softmax(x, dim=1)
        
        return output

# 모델 인스턴스 생성
model = CNN()
print(model)

# 테스트용 더미 입력 데이터 생성 (Batch Size: 1, Channels: 1, Height: 28, Width: 28)
dummy_input = torch.randn(1, 1, 28, 28)
output = model(dummy_input)

print("\n출력 결과 형태 (Shape):", output.shape)
print("출력 확률 값:", output)

CNN(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=784, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=10, bias=True)
)

출력 결과 형태 (Shape): torch.Size([1, 10])
출력 확률 값: tensor([[0.0807, 0.0824, 0.0992, 0.1111, 0.0909, 0.1086, 0.1162, 0.0942, 0.1172,
         0.0995]], grad_fn=<SoftmaxBackward0>)
